# FoodVision AI — Google Colab runner
This notebook runs the **GitHub-ready** project without repeating training.
Select **Runtime → Change runtime type → T4 GPU**. Upload `FoodVision_GitHub_Ready.zip` and your private `food_dataset.zip`.
To run directly from your published GitHub repository instead, clone your repository into Colab and follow the README.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert "FoodVision_GitHub_Ready.zip" in uploaded, "Upload the GitHub-ready ZIP"
assert "food_dataset.zip" in uploaded, "Upload your original private food dataset ZIP"


In [ ]:
!mkdir -p /content/FoodVision-Assignment2
!unzip -oq /content/FoodVision_GitHub_Ready.zip -d /content/FoodVision-Assignment2
%cd /content/FoodVision-Assignment2
!pip install -q -r requirements.txt
!python prepare_data.py --zip /content/food_dataset.zip
!python audit_dataset.py


In [ ]:
import json, torch
from pathlib import Path
print("CUDA available:", torch.cuda.is_available())
m = json.loads(Path("artifacts/metrics.json").read_text())
print("Trained model results:", m["selected"])
print("Model is already trained: no need to rerun training.")


## Optional: reproduce training
Only run the next cell if you need to **retrain** from scratch. It overwrites the saved trained head and results.


In [ ]:
# Uncomment to retrain; the shipped classifier is already usable.
#!python train.py --data-dir food_dataset --output-dir artifacts --epochs 50 --batch-size 16


## Run the Streamlit demo from Colab
The final cell starts a temporary public tunnel. It **keeps running** while the website is reachable. The temporary URL changes between sessions.


In [ ]:
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/foodvision-streamlit.log 2>&1 &
!/tmp/cloudflared tunnel --url http://localhost:8501
